# Hypothesis Testing - Exercises

Practice problems for hypothesis testing concepts.

## Exercises
1. One-Sample t-Test
2. Two-Sample t-Test (Model Comparison)
3. Paired t-Test (Before/After)
4. Chi-Square Test

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)

---
## Exercise 1: One-Sample t-Test

**Scenario**: A model is claimed to have 90% accuracy. You run 12 experiments and get these results.

**Tasks**:
1. State H₀ and H₁
2. Calculate the test statistic manually
3. Find the p-value
4. Make a decision at α = 0.05

In [ ]:
# Experiment results (% accuracy)
results = np.array([88.5, 91.2, 87.3, 89.8, 92.1, 88.0, 90.5, 86.9, 89.2, 91.0, 88.7, 89.5])
claimed_accuracy = 90

# Your solution here

### Solution

In [ ]:
# Experiment results (% accuracy)
results = np.array([88.5, 91.2, 87.3, 89.8, 92.1, 88.0, 90.5, 86.9, 89.2, 91.0, 88.7, 89.5])
claimed_accuracy = 90

print("ONE-SAMPLE t-TEST")
print("="*55)

# 1. State hypotheses
print("\n1️⃣ Hypotheses:")
print(f"  H₀: μ = {claimed_accuracy}% (accuracy equals claim)")
print(f"  H₁: μ ≠ {claimed_accuracy}% (accuracy differs from claim)")

# 2. Calculate test statistic
n = len(results)
x_bar = results.mean()
s = results.std(ddof=1)
se = s / np.sqrt(n)
t_stat = (x_bar - claimed_accuracy) / se

print("\n2️⃣ Test Statistic:")
print(f"  n = {n}")
print(f"  x̄ = {x_bar:.4f}")
print(f"  s = {s:.4f}")
print(f"  SE = s/√n = {se:.4f}")
print(f"  t = (x̄ - μ₀)/SE = ({x_bar:.4f} - {claimed_accuracy})/{se:.4f}")
print(f"  t = {t_stat:.4f}")

# 3. Find p-value
df = n - 1
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df))  # Two-tailed

print(f"\n3️⃣ P-value:")
print(f"  df = n - 1 = {df}")
print(f"  p-value (two-tailed) = {p_value:.6f}")

# Verify with scipy
t_scipy, p_scipy = stats.ttest_1samp(results, claimed_accuracy)
print(f"  ✓ scipy check: t = {t_scipy:.4f}, p = {p_scipy:.6f}")

# 4. Decision
alpha = 0.05
print(f"\n4️⃣ Decision (α = {alpha}):")
if p_value < alpha:
    print(f"  p = {p_value:.4f} < α = {alpha}")
    print("  ✅ REJECT H₀")
    print(f"  Conclusion: Accuracy is significantly different from {claimed_accuracy}%")
else:
    print(f"  p = {p_value:.4f} ≥ α = {alpha}")
    print("  ❌ FAIL TO REJECT H₀")
    print(f"  Conclusion: No significant difference from {claimed_accuracy}%")

# Confidence interval
t_crit = stats.t.ppf(1 - alpha/2, df)
ci = (x_bar - t_crit * se, x_bar + t_crit * se)
print(f"\n📊 95% CI: [{ci[0]:.2f}, {ci[1]:.2f}]")
print(f"   Does CI include {claimed_accuracy}? {ci[0] <= claimed_accuracy <= ci[1]}")

---
## Exercise 2: Two-Sample t-Test (Model Comparison)

**Scenario**: Compare ResNet vs VGG on the same dataset. Each model was trained 15 times with different seeds.

**Tasks**:
1. Perform Welch's t-test (unequal variances)
2. Calculate Cohen's d effect size
3. Make a decision and interpret

In [ ]:
# F1 scores from 15 runs each
np.random.seed(42)
resnet_scores = np.random.normal(0.892, 0.025, 15)
vgg_scores = np.random.normal(0.875, 0.035, 15)

# Your solution here

### Solution

In [ ]:
# F1 scores from 15 runs each
np.random.seed(42)
resnet_scores = np.random.normal(0.892, 0.025, 15)
vgg_scores = np.random.normal(0.875, 0.035, 15)

print("TWO-SAMPLE t-TEST: ResNet vs VGG")
print("="*55)

print("\n📊 Sample Statistics:")
print(f"  ResNet: n={len(resnet_scores)}, mean={resnet_scores.mean():.4f}, std={resnet_scores.std(ddof=1):.4f}")
print(f"  VGG:    n={len(vgg_scores)}, mean={vgg_scores.mean():.4f}, std={vgg_scores.std(ddof=1):.4f}")

# 1. Welch's t-test
t_stat, p_value = stats.ttest_ind(resnet_scores, vgg_scores, equal_var=False)

print("\n1️⃣ Welch's t-test (unequal variances):")
print(f"  t = {t_stat:.4f}")
print(f"  p-value = {p_value:.6f}")

# 2. Cohen's d
pooled_std = np.sqrt((resnet_scores.var(ddof=1) + vgg_scores.var(ddof=1)) / 2)
cohens_d = (resnet_scores.mean() - vgg_scores.mean()) / pooled_std

print("\n2️⃣ Effect Size (Cohen's d):")
print(f"  Pooled std = {pooled_std:.4f}")
print(f"  d = (μ₁ - μ₂) / s_pooled = {cohens_d:.4f}")
print("  Interpretation:")
if abs(cohens_d) < 0.2:
    print("    |d| < 0.2 → Small effect")
elif abs(cohens_d) < 0.8:
    print("    0.2 ≤ |d| < 0.8 → Medium effect")
else:
    print("    |d| ≥ 0.8 → Large effect")

# 3. Decision
alpha = 0.05
print(f"\n3️⃣ Decision (α = {alpha}):")
if p_value < alpha:
    print(f"  p = {p_value:.6f} < {alpha}")
    print("  ✅ REJECT H₀")
    winner = "ResNet" if resnet_scores.mean() > vgg_scores.mean() else "VGG"
    diff = abs(resnet_scores.mean() - vgg_scores.mean()) * 100
    print(f"  Conclusion: {winner} performs significantly better!")
    print(f"  Practical difference: ~{diff:.2f}% in F1 score")
else:
    print(f"  p = {p_value:.6f} ≥ {alpha}")
    print("  ❌ FAIL TO REJECT H₀")
    print("  Conclusion: No significant difference between models")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot([resnet_scores, vgg_scores], labels=['ResNet', 'VGG'])
ax.scatter([1]*len(resnet_scores), resnet_scores, alpha=0.5, color='blue')
ax.scatter([2]*len(vgg_scores), vgg_scores, alpha=0.5, color='orange')
ax.set_ylabel('F1 Score')
ax.set_title(f'Model Comparison (p={p_value:.4f}, d={cohens_d:.3f})')
plt.show()

---
## Exercise 3: Paired t-Test (Before/After)

**Scenario**: You applied data augmentation to improve a model. Same 10 test datasets, before and after.

**Tasks**:
1. Calculate the differences
2. Perform paired t-test
3. Compare with independent t-test (explain why paired is better)

In [ ]:
# Accuracy (%) on 10 datasets
before_aug = np.array([78.2, 82.5, 75.1, 88.3, 71.2, 79.8, 84.5, 76.3, 80.1, 85.2])
after_aug = np.array([80.1, 84.2, 77.8, 89.5, 74.5, 82.1, 86.0, 78.9, 82.3, 86.8])

# Your solution here

### Solution

In [ ]:
# Accuracy (%) on 10 datasets
before_aug = np.array([78.2, 82.5, 75.1, 88.3, 71.2, 79.8, 84.5, 76.3, 80.1, 85.2])
after_aug = np.array([80.1, 84.2, 77.8, 89.5, 74.5, 82.1, 86.0, 78.9, 82.3, 86.8])

print("PAIRED t-TEST: Data Augmentation Effect")
print("="*55)

# 1. Calculate differences
differences = after_aug - before_aug

print("\n1️⃣ Paired Differences:")
print(f"  Before: {before_aug}")
print(f"  After:  {after_aug}")
print(f"  Diff:   {differences}")
print(f"  ")
print(f"  Mean difference: {differences.mean():.4f}%")
print(f"  Std of differences: {differences.std(ddof=1):.4f}%")

# 2. Paired t-test
t_paired, p_paired = stats.ttest_rel(before_aug, after_aug)

print("\n2️⃣ Paired t-test:")
print(f"  t = {t_paired:.4f}")
print(f"  p-value = {p_paired:.6f}")

# 3. Compare with independent t-test
t_indep, p_indep = stats.ttest_ind(before_aug, after_aug)

print("\n3️⃣ Independent t-test (WRONG for this data):")
print(f"  t = {t_indep:.4f}")
print(f"  p-value = {p_indep:.6f}")

print("\n📊 Comparison:")
print(f"  Paired p-value:      {p_paired:.6f}")
print(f"  Independent p-value: {p_indep:.6f}")
print(f"  Ratio: {p_indep/p_paired:.2f}x difference")

print("\n❓ Why paired is better here:")
print("  - Each before/after pair uses the SAME dataset")
print("  - Datasets have different difficulty (71% to 88%)")
print("  - Paired test removes between-dataset variability")
print("  - Only focuses on the consistent improvement")
print(f"  - Std of differences ({differences.std(ddof=1):.2f}) << Std of each group (~5)")

# Decision
alpha = 0.05
print(f"\n🎯 Decision (α = {alpha}):")
if p_paired < alpha:
    print(f"  p = {p_paired:.6f} < {alpha}")
    print("  ✅ REJECT H₀")
    print(f"  Data augmentation significantly improves accuracy by ~{differences.mean():.1f}%!")
else:
    print(f"  p = {p_paired:.6f} ≥ {alpha}")
    print("  ❌ FAIL TO REJECT H₀")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Connected lines
for i in range(len(before_aug)):
    axes[0].plot([0, 1], [before_aug[i], after_aug[i]], 'o-', alpha=0.6)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Before', 'After'])
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Paired Data: Each Line is One Dataset')

# Differences
axes[1].bar(range(len(differences)), differences, color='green', alpha=0.7)
axes[1].axhline(differences.mean(), color='red', linestyle='--', label=f'Mean = {differences.mean():.2f}')
axes[1].set_xlabel('Dataset')
axes[1].set_ylabel('Improvement (After - Before)')
axes[1].set_title('Consistent Positive Improvement')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Exercise 4: Chi-Square Test

**Scenario**: Test if there's a relationship between model architecture and task performance category.

**Tasks**:
1. Perform chi-square test of independence
2. Calculate expected frequencies
3. Calculate Cramér's V effect size
4. Interpret the results

In [ ]:
# Contingency table
#                 Good (>80%)  Medium (60-80%)  Poor (<60%)
# Transformer          45             30            25
# CNN                   35             40            25
# RNN                   20             30            50

contingency = np.array([
    [45, 30, 25],  # Transformer
    [35, 40, 25],  # CNN
    [20, 30, 50]   # RNN
])

# Your solution here

### Solution

In [ ]:
# Contingency table
contingency = np.array([
    [45, 30, 25],  # Transformer
    [35, 40, 25],  # CNN
    [20, 30, 50]   # RNN
])

print("CHI-SQUARE TEST OF INDEPENDENCE")
print("="*55)

print("\n📊 Observed Frequencies:")
print("              Good(>80%)  Medium(60-80%)  Poor(<60%)  Total")
row_names = ['Transformer', 'CNN        ', 'RNN        ']
for i, name in enumerate(row_names):
    print(f"{name}    {contingency[i,0]:5d}       {contingency[i,1]:5d}        {contingency[i,2]:5d}     {contingency[i].sum():5d}")
print(f"Total          {contingency[:,0].sum():5d}       {contingency[:,1].sum():5d}        {contingency[:,2].sum():5d}     {contingency.sum():5d}")

# 1. Chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print(f"\n1️⃣ Chi-Square Test:")
print(f"  χ² = {chi2:.4f}")
print(f"  df = {dof}")
print(f"  p-value = {p_value:.6f}")

# 2. Expected frequencies
print(f"\n2️⃣ Expected Frequencies (if independent):")
print("              Good(>80%)  Medium(60-80%)  Poor(<60%)")
for i, name in enumerate(row_names):
    print(f"{name}    {expected[i,0]:5.1f}       {expected[i,1]:5.1f}        {expected[i,2]:5.1f}")

# 3. Cramér's V
n = contingency.sum()
min_dim = min(contingency.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

print(f"\n3️⃣ Effect Size (Cramér's V):")
print(f"  V = √(χ²/(n × min(r-1, c-1)))")
print(f"  V = √({chi2:.2f}/({n} × {min_dim}))")
print(f"  V = {cramers_v:.4f}")
print("  Interpretation:")
if cramers_v < 0.1:
    print("    V < 0.1 → Negligible effect")
elif cramers_v < 0.2:
    print("    0.1 ≤ V < 0.2 → Weak effect")
elif cramers_v < 0.4:
    print("    0.2 ≤ V < 0.4 → Moderate effect")
else:
    print("    V ≥ 0.4 → Strong effect")

# 4. Interpretation
alpha = 0.05
print(f"\n4️⃣ Interpretation (α = {alpha}):")
if p_value < alpha:
    print(f"  p = {p_value:.6f} < {alpha}")
    print("  ✅ REJECT H₀")
    print("  Architecture and performance ARE related!")
    print("")
    print("  📊 Insights:")
    print("  - Transformers: More 'Good' than expected")
    print("  - RNNs: More 'Poor' than expected")
    print("  - CNNs: Fairly balanced distribution")
else:
    print(f"  p = {p_value:.6f} ≥ {alpha}")
    print("  ❌ FAIL TO REJECT H₀")
    print("  No evidence of relationship")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(3)
width = 0.25

ax.bar(x - width, contingency[0], width, label='Transformer', color='#3498db')
ax.bar(x, contingency[1], width, label='CNN', color='#2ecc71')
ax.bar(x + width, contingency[2], width, label='RNN', color='#e74c3c')

ax.set_xticks(x)
ax.set_xticklabels(['Good (>80%)', 'Medium (60-80%)', 'Poor (<60%)'])
ax.set_ylabel('Count')
ax.set_title(f'Architecture vs Performance (χ²={chi2:.2f}, p={p_value:.4f}, V={cramers_v:.3f})')
ax.legend()
plt.tight_layout()
plt.show()